# Ch2 Embedding 教案：语言模型的灵魂

---

## 课程信息

| 项目 | 内容 |
|:---|:---|
| **课程标题** | Ch2 Embedding：语言模型的灵魂 |
| **预计总时长** | 90 分钟（含休息） |
| **源文件** | `Ch2_Embedding/Ch2_Embedding.ipynb` |
| **前置章节** | Ch1 Autograd（梯度与反向传播） |
| **下一章** | Ch3 Self-Attention |

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 00:00-05:00 | 开场导入 + 环境准备 | Cell 0-4 | 5 分钟 |
| 05:00-20:00 | S1 Embedding 本质：查表矩阵 | Cell 5-8 | 15 分钟 |
| 20:00-45:00 | S2 Bigram 模型训练词向量 | Cell 9-14 | 25 分钟 |
| 45:00-50:00 | 课间休息 | - | 5 分钟 |
| 50:00-60:00 | S3 可视化词向量空间 | Cell 15-17 | 10 分钟 |
| 60:00-75:00 | S4 余弦相似度 + 练习 | Cell 18-20 | 15 分钟 |
| 75:00-85:00 | S5 One-Hot vs Embedding 对比 + 总结 | Cell 21-26 | 10 分钟 |
| 85:00-90:00 | 答疑 + 下一章预告 | Cell 26-27 | 5 分钟 |

---

## 课前准备

- [ ] 确认 Python 环境已安装 PyTorch、sklearn、matplotlib、numpy
- [ ] 确认中文字体可正常显示（Microsoft YaHei 或 SimHei）
- [ ] 提前运行一遍源 notebook，确认所有 Cell 可正常执行（约 0.22 秒训练）
- [ ] 准备好备用方案：若字体路径 `../assets/fonts/NotoSansCJKsc-Regular.otf` 不存在，plt.rcParams 中的备选字体需可用
- [ ] 检查投影仪分辨率，确认 `figure.dpi = 120` 的图像大小合适
- [ ] 准备白板或画板，用于手绘查表矩阵示意图


## 段落 1：开场导入 + 环境准备

📍 运行 Cell 0-4（标题、学习路线、前置知识、环境准备代码）

⏱ 时间分配：5 分钟

---

### 🎯 本段目标

- 激发学生兴趣：机器如何"理解"文字？
- 明确本章三大学习目标：查表矩阵 -> 训练词向量 -> 可视化
- 确认环境就绪（PyTorch 版本输出）

---

### 🗣 讲课话术

> 大家好！上一章我们学了 Autograd——让 PyTorch 自动帮我们求梯度。今天我们要解决一个更根本的问题：**机器怎么读懂中文？**
>
> 你想想，计算机本质上只认识 0 和 1。你给它一个"猫"字，它完全不知道这是什么。所以我们需要一个桥梁，把文字变成数字——而且是**有意义的数字**。
>
> 最笨的方法是 One-Hot：词表里有多少词就弄多少维，"猫"是 `[1,0,0]`，"狗"是 `[0,1,0]`。但问题来了——"猫"和"狗"的距离跟"猫"和"汽车"的距离一模一样！这不科学吧？
>
> 今天我们要学的 Embedding，就是要解决这个问题。先来确认大家的环境能跑通。

**运行 Cell 4**，确认输出：`PyTorch version: 2.10.0+cu128`（或当前版本号）。

---

### 👀 输出要点

- Cell 4 应输出 `PyTorch version: 2.10.0+cu128`
- 若字体相关 warning 出现，可忽略，不影响后续运行

---

### ❓ 预判问题

**Q: 为什么用按字符分词而不是用 jieba 分词？**

A: 本章目的是演示 Embedding 原理，用字符分词最简单直观，避免分词工具带来的额外复杂度。实际生产中会用更好的 tokenizer（如 BPE）。

**Q: 这些前置知识我没学过怎么办？**

A: 核心需要理解的是"训练循环"（forward -> loss -> backward -> step）和"梯度"的概念。如果不熟悉，建议课后补 Ch0 和 Ch1。

---

### ➡️ 转场

> 好，环境没问题。现在让我们揭开 Embedding 的面纱——它其实简单到令人发指。


## 段落 2：S1 Embedding 的本质——查表矩阵

📍 运行 Cell 5-8（理论讲解 + 创建 Embedding 层 + 查表验证）

⏱ 时间分配：15 分钟

---

### 🎯 本段目标

- 理解 `nn.Embedding` 就是一个 `[vocab_size, embed_dim]` 的矩阵
- 理解"查表"操作：输入索引 -> 输出对应行向量
- 验证 `embedding(idx)` 和 `embedding.weight[idx]` 完全等价

---

### 🗣 讲课话术

> 忘掉你听过的所有高大上的定义。Embedding 的本质就四个字：**查、表、矩、阵**。
>
> 想象一下你有一本字典，每个字对应一页。你查"猫"这个字，翻到第 5 页，那一页上写着一串数字 `[0.2, 0.8, -0.3, 0.5]`。这就是 Embedding 做的事。
>
> **运行 Cell 6** 看看——我们创建了一个 10 个词、每个词 4 维的 Embedding 层。看输出：矩阵形状是 `torch.Size([10, 4])`，就是一个 10 行 4 列的矩阵，每一行对应一个词的向量。这些数值是随机初始化的，训练之前毫无意义。
>
> 注意看第 0 行的数值：`[1.9269, 1.4873, 0.9007, -2.1055]`。现在这些数字完全没有语义含义——等我们训练之后就不一样了。
>
> 现在关键来了——**运行 Cell 7**。我们查询词索引 `[0, 3, 5]`，用两种方式：
> - `embedding(word_idx)`：PyTorch 的标准用法
> - `embedding.weight[word_idx]`：直接索引矩阵
>
> 看最后一行输出：`两种方法完全等价: True`。这就证明了 Embedding **就是查表**，没有任何额外的计算。
>
> 但请注意一个细节——方法 1 的 `grad_fn` 是 `EmbeddingBackward0`，方法 2 是 `IndexBackward0`。虽然前向结果一样，但 PyTorch 记录的计算图不同。用 `nn.Embedding` 的好处是它会正确处理梯度的稀疏更新。

---

### 👀 输出要点

- Cell 6：矩阵形状 `torch.Size([10, 4])`，10 行 4 列
- Cell 6：矩阵值为随机初始化，例如第 0 行 `[1.9269, 1.4873, 0.9007, -2.1055]`
- Cell 7：查询索引 `[0, 3, 5]`，返回 3 个 4 维向量
- Cell 7：`torch.allclose` 返回 `True`，证明两种方法等价
- Cell 7：`grad_fn` 分别为 `EmbeddingBackward0` 和 `IndexBackward0`

---

### ❓ 预判问题

**Q: 为什么 Embedding 矩阵是随机初始化的？**

A: 跟神经网络的权重一样，随机初始化提供一个起点，然后通过训练（梯度下降）不断调整。训练完成后这些数值才有语义含义。

**Q: One-hot 乘以 Embedding 矩阵不也是取出一行吗？为什么不直接做矩阵乘法？**

A: 数学上完全等价！但矩阵乘法是 O(V*d)，而直接索引是 O(d)。当词表有 5 万个词时，查表比矩阵乘法快 5 万倍。Cell 5 的理论部分有详细解释。

**Q: embed_dim 怎么选？4 维够吗？**

A: 这里 4 维只是为了演示。实际中常用 256、512、768 或 1024 维。GPT-3 用的是 12288 维。维度越高能编码越多信息，但计算成本也越高。

---

### ➡️ 转场

> 好，现在我们知道 Embedding 就是查表。但随机初始化的表毫无用处——"猫"的向量和"狗"的向量一样随机。关键问题是：**怎么让这个表变得有意义？** 答案是——训练！


## 段落 3：S2 训练词向量——Bigram 语言模型

📍 运行 Cell 9-14（理论 + 数据准备 + 模型定义 + 训练 + Loss 曲线）

⏱ 时间分配：25 分钟

---

### 🎯 本段目标

- 理解 Bigram 模型的结构：Embedding -> Linear -> Softmax
- 理解"通过预测下一个词来训练词向量"的核心思想
- 能读懂训练循环并理解 Loss 的下降趋势

---

### 🗣 讲课话术

> 这一节是本章最核心的部分。我们要亲手训练一个模型，让它学会"给定一个字，预测下一个字"。副产品就是有意义的词向量。
>
> **先看数据——运行 Cell 10**。训练文本很短，只有 5 句话，总共 52 个字符，去重后词表只有 28 个字。看看词表输出：`['世', '习', '人', '分', '变', '器', ...]`。
>
> **运行 Cell 11** 构建训练数据。我们把文本变成 51 个"当前字->下一个字"的配对。看输出前 5 个样本：`机->器`、`器->学`、`学->习`、`习->是`、`是->人`。这就是 Bigram——只看一个字预测下一个字。
>
> 大家注意一个有趣的细节：为什么是 51 个样本而不是 52 个？因为最后一个字没有"下一个字"可以预测。所以 51 = 52 - 1。
>
> **运行 Cell 12** 看模型结构。`BigramModel` 只有两层：
> - `embedding`: Embedding(28, 16)——28 个字，每个用 16 维向量表示
> - `linear`: Linear(16, 28)——把 16 维向量映射回 28 个字的概率
>
> 总参数量才 924 个！怎么算的？Embedding: 28*16=448，Linear 权重: 16*28=448，Linear 偏置: 28，加起来 448+448+28=924。对比一下，GPT-3 有 1750 亿个参数，差了 1.9 亿倍。
>
> **关键时刻——运行 Cell 13** 开始训练！看 Loss 的变化：
> - Epoch 100：Loss = 0.3044
> - Epoch 200：Loss = 0.2924
> - Epoch 300：Loss = 0.2897
> - Epoch 500：Loss = 0.2881
>
> Loss 从高位快速下降，然后逐渐趋于平稳。0.22 秒就训练完了——因为数据太小了。
>
> **运行 Cell 14** 看 Loss 曲线图。注意前 50 个 epoch 下降最快，后面基本收敛。最终损失标注在图上：`最终损失: 0.2881`。
>
> 大家想一想：如果这个 Loss 等于 0 意味着什么？意味着模型完美预测了每一个下一个字。但我们的数据里"学"后面既可以是"习"也可以是"的"，所以 Loss 不可能到 0——这是数据本身的不确定性，叫做数据的**内在熵**。

---

### 👀 输出要点

- Cell 10：总词数 52，词表大小 28
- Cell 11：训练样本数 51，前 5 个样本 `机->器, 器->学, 学->习, 习->是, 是->人`
- Cell 12：模型总参数量 924（Embedding 448 + Linear权重 448 + 偏置 28）
- Cell 13：Epoch 100 Loss=0.3044，Epoch 500 Loss=0.2881，训练耗时 0.22 秒
- Cell 14：Loss 曲线前 50 epoch 下降最快，之后基本收敛

---

### ❓ 预判问题

**Q: 为什么用 Adam 优化器而不是 SGD？**

A: Adam 对学习率不太敏感，收敛更快。对于这种小规模实验，Adam 的自适应学习率特别方便。SGD 也能训练，但可能需要更仔细地调参。

**Q: CrossEntropyLoss 和 Softmax 是什么关系？**

A: `nn.CrossEntropyLoss` 内部已经包含了 Softmax 操作（更准确地说是 log_softmax + NLLLoss）。所以模型输出的是 logits（未归一化的分数），不需要自己加 Softmax。

**Q: 为什么 Loss 不能降到 0？**

A: 因为训练数据中同一个字后面可能接不同的字（如"学"后面可以是"习"也可以是"的"），模型没法同时以 100% 概率预测两个不同的字。这叫做数据的**内在熵**。

**Q: 924 个参数是怎么算出来的？**

A: Embedding: 28*16 = 448；Linear 权重: 16*28 = 448；Linear 偏置: 28。合计 448+448+28 = 924。

---

### ➡️ 转场

> 训练完成了！现在这个 Embedding 矩阵里的数值已经不再是随机的了——它们承载了从预测任务中学到的语义信息。下面让我们亲眼看看这些词向量在空间里长什么样。


## 课间休息（5 分钟）

---

### 前半段回顾（3 句话）

1. **Embedding 就是查表**：`nn.Embedding` 是一个 `[vocab_size, embed_dim]` 的可学习矩阵，输入索引返回对应行向量。
2. **训练让查表有意义**：通过 Bigram 预测任务（给定当前字预测下一个字），Embedding 矩阵中的向量被迫编码语义关系。
3. **924 个参数、0.22 秒训练**：我们用一个极简模型演示了预训练的核心思想——简单的预测任务能产生有用的表示。

---

### 下半段预告

- 用 PCA 可视化 28 个字的词向量分布
- 实现余弦相似度，量化词与词的相似程度
- 对比 One-Hot 和 Embedding 的维度压缩效果


## 段落 4：S3 可视化词向量空间

📍 运行 Cell 15-17（理论 + PCA 降维 + 散点图）

⏱ 时间分配：10 分钟

---

### 🎯 本段目标

- 理解 PCA 降维的作用：16 维 -> 2 维以便画图
- 在散点图中观察语义相近的字是否聚在一起
- 理解 PCA 解释方差比的含义

---

### 🗣 讲课话术

> 16 维空间我们没法直接看到，所以用 PCA 把它压缩到 2 维来画图。PCA 做的事情就是找到数据变化最大的两个方向，把高维点投影上去。
>
> **运行 Cell 16**——词向量形状是 `(28, 16)`，28 个字各 16 维。PCA 解释方差比是 `[0.157, 0.132]`，加起来约 29%。意思是这两个维度只保留了约 29% 的信息——信息丢失不少，所以图上的位置不要过度解读。
>
> **运行 Cell 17** 看散点图。大家观察几个现象：
> - "机"和"器"是不是挨得比较近？因为训练数据里"机器"总是连着出现
> - "学"和"习"呢？"学习"也是高频共现对
> - "人"和"工"呢？"人工"也总一起出现
>
> 这就是词向量的魔力——**我们从来没有告诉模型这些字有关系**，模型通过预测下一个字的任务，自己发现了这些关系！
>
> 当然，因为我们的训练数据只有 5 句话、52 个字，所以聚类效果不会太完美。如果用百万级别的语料训练，效果会好得多。

---

### 👀 输出要点

- Cell 16：词向量形状 `(28, 16)`
- Cell 16：PCA 解释方差比 `[0.157, 0.132]`，合计约 29%
- Cell 17：散点图中每个字标注在对应位置
- Cell 17：左上角标注了 PCA1 和 PCA2 的解释方差（15.7% 和 13.2%）

---

### ❓ 预判问题

**Q: PCA 解释方差比只有 29%，这个图可靠吗？**

A: 只能说"大致参考"。29% 意味着 71% 的信息丢失了。两个在 16 维空间中近的点，在 2D 图上可能看起来远；反之亦然。但大趋势是有参考价值的。

**Q: 为什么有些字看起来离得很远但它们语义相关？**

A: 两个原因：(1) PCA 降维丢失了大量信息；(2) 我们的训练数据太少（只有 52 个字），模型还没有充分学习。

**Q: t-SNE 是不是比 PCA 更好？**

A: t-SNE 更擅长保持局部邻域关系，对于词向量可视化通常效果更好。但 PCA 更简单、确定性强（不像 t-SNE 每次结果可能不同），适合教学演示。

---

### ➡️ 转场

> 可视化给了我们直觉，但直觉不够精确。我们需要一个数字来量化两个词向量有多相似——这就是余弦相似度。而且这里还有一个动手练习等着大家！


## 段落 5：S4 余弦相似度 + 练习

📍 运行 Cell 18-20（余弦相似度理论 + 代码实现练习 + 相似度矩阵可视化）

⏱ 时间分配：15 分钟

---

### 🎯 本段目标

- 理解余弦相似度的公式和几何意义
- （练习）完成 `cosine_similarity` 函数中的 3 个 TODO
- 使用 `find_similar()` 查找相似词并解读结果
- 阅读相似度热力图

---

### 🗣 讲课话术

> 余弦相似度衡量的是两个向量的**方向**有多一致，不关心长度。公式很简单：点积除以两个模长的乘积。
>
> 如果两个向量方向完全一样（夹角 0 度），余弦值是 1；完全垂直（90 度）是 0；完全反向（180 度）是 -1。
>
> Cell 18 的理论部分有一个很好的例子：向量 A=(3,4) 和 B=(4,3)，余弦相似度是 24/25 = 0.96，非常相似。
>
> 现在到了动手时间——**看 Cell 19**。`cosine_similarity` 函数里有 3 个 TODO 需要大家填写。给大家 4 分钟时间。

---

### 练习引导：Cell 19 实现 cosine_similarity 函数

| 时间 | 引导 |
|:---|:---|
| 0-2 分钟 | 自己尝试。提示：需要用到 `np.dot()` 和 `np.linalg.norm()` |
| 2 分钟 | 第一个提示：Step 1 是 `np.dot(v1, v2)`，Step 2 是对 v1 和 v2 分别求 `np.linalg.norm()` |
| 4 分钟 | 公布答案：`dot_product = np.dot(v1, v2)`, `norm_v1 = np.linalg.norm(v1)`, `norm_v2 = np.linalg.norm(v2)`, `similarity = dot_product / (norm_v1 * norm_v2)` |

**常见错误：**

- 忘记分别对两个向量求 norm（只求了一个）
- 点积和模长搞混（把 `np.linalg.norm` 用在两个向量拼起来的结果上）
- 除法分母少乘一个 norm

**验证标准：**

- `cos_sim(v, v) = 1.0000`（应为 1.0）
- `cos_sim(v, -v) = -1.0000`（应为 -1.0）
- 如果这两个验证通过，说明实现正确

---

### 🗣 练习完成后的讲解话术

> 运行 Cell 19，看验证结果。`cos_sim(v, v) = 1.0000`，`cos_sim(v, -v) = -1.0000`，完美！
>
> 然后看 `find_similar('学')` 的输出：最相似的是"界"(0.2223)、"神"(0.2058)、"基"(0.1862)。跟我们的直觉不太一样对吧？"习"只排第 5（0.0971）。这是因为训练数据太少了，而且"学"后面不只接"习"，还接了其他字。
>
> 再看 `find_similar('机')`：最相似的是"工"(0.3525)。这倒是合理的——"机器"和"人工"在训练数据中都出现了多次，"机"和"工"共享了相似的上下文。
>
> **运行 Cell 20** 看整个相似度矩阵的热力图。对角线全是 1.0（自己和自己完全相同）。红色越深表示越相似，蓝色越深表示越不相似。每个格子里标注了具体数值，大家可以找找有没有特别高或特别低的区域。

---

### 👀 输出要点

- Cell 19 验证：`cos_sim(v, v) = 1.0000`，`cos_sim(v, -v) = -1.0000`
- Cell 19 `find_similar('学')`：界(0.2223), 神(0.2058), 基(0.1862), 器(0.1546), 习(0.0971)
- Cell 19 `find_similar('机')`：工(0.3525), 础(0.3290), 活(0.2700), 神(0.2671), 生(0.1889)
- Cell 20：28x28 热力图，对角线为 1.0，颜色从蓝(-1)到红(+1)

---

### ❓ 预判问题

**Q: 为什么相似度普遍不高？最高才 0.35？**

A: 训练数据只有 52 个字，模型没法学到足够丰富的语义关系。用真实大规模语料训练的 Word2Vec，"猫"和"狗"的余弦相似度通常在 0.7-0.8 之间。

**Q: 为什么用余弦相似度而不是欧氏距离？**

A: 余弦相似度只关注方向，不受向量长度影响。Cell 18 理论部分有详细对比：如果一个向量是另一个的 100 倍缩放，点积和欧氏距离会给出误导性结果，但余弦相似度仍然是 1.0。

**Q: 为什么 find_similar 的结果跟 PCA 图上的位置不完全对应？**

A: PCA 图是 16 维降到 2 维的投影，丢失了 71% 的信息。余弦相似度是在完整的 16 维空间计算的，更准确。

---

### ➡️ 转场

> 现在我们对词向量有了数量化的理解。最后让我们回到原点，正式对比一下 One-Hot 和 Embedding 这两种表示方式。


## 段落 6：S5 One-Hot vs Embedding 对比 + 本章总结

📍 运行 Cell 21-26（One-Hot 对比 + 总结 + 延伸阅读）

⏱ 时间分配：10 分钟

---

### 🎯 本段目标

- 通过具体数值对比 One-Hot 和 Embedding 的差异
- 串联本章所有知识点
- 留下延伸探索方向

---

### 🗣 讲课话术

> **运行 Cell 22**。看词"分"（索引 3）的两种表示：
> - One-Hot：28 维，里面只有 1 个 1，其余全是 0。极度浪费！
> - Embedding：16 维，每个维度都有数值，例如 `[-0.8817, -0.1062, 0.8970, ...]`。稠密、紧凑、有意义。
>
> 维度压缩比是 28/16 = 1.8 倍。看起来不多对吧？那是因为我们词表只有 28 个字。如果词表有 5 万个字（实际 NLP 系统的典型大小），One-Hot 是 5 万维，Embedding 可能只需要 768 维，压缩比超过 60 倍！
>
> Cell 23 有一个很好的对比表格，大家可以截图留存。关键的五个维度：维度大小、稀疏性、语义关系、可学习性、内存占用。
>
> 现在让我们回顾一下整章的核心概念——看 Cell 24 的总结。三个要点：
> 1. **Embedding 就是查表**：`nn.Embedding` = 一个可学习的矩阵
> 2. **训练让查表有意义**：通过预测任务自动学到语义关系
> 3. **余弦相似度量化语义**：方向相同 -> 语义相近
>
> Cell 24 还有 3 个常见面试题，大家可以看看自己能不能回答。
>
> Cell 25 列了几个课后探索方向，感兴趣的同学可以试试。特别推荐"改变嵌入维度"——把 embed_dim 从 16 改成 4 或 64，看看对相似度结果的影响。

---

### 👀 输出要点

- Cell 22：词"分"的 One-Hot 为 28 维稀疏向量，Embedding 为 16 维稠密向量
- Cell 22：维度压缩比 1.8x（小词表场景；实际大词表可达 60x+）
- Cell 24：核心公式速查表（Embedding 查表、余弦相似度、Bigram 概率、交叉熵、向量距离）
- Cell 24：3 个常见面试题及参考答案

---

### ❓ 预判问题

**Q: 为什么 GPT 不直接用 Word2Vec 的词向量？**

A: GPT 等 Transformer 模型会从头训练自己的 Embedding，因为它们的 Embedding 会与后续的 Attention 层协同优化。静态词向量（如 Word2Vec）无法处理一词多义——"苹果"作为水果和公司应该有不同的向量，但 Word2Vec 只给一个。这正是下一章 Self-Attention 要解决的问题。

**Q: 在实际项目中 Embedding 维度一般多大？**

A: 经典 Word2Vec 常用 100-300 维。BERT-base 用 768 维，GPT-2 用 768 维，GPT-3 用 12288 维。一般来说，数据量越大、模型越复杂，嵌入维度越高。

---

### ➡️ 转场

> 今天我们从"机器怎么读懂中文"这个问题出发，一步步理解了 Embedding 的本质、训练方法、可视化和度量方式。下一章我们将看到 Embedding 的一个重大局限——它是静态的，同一个字永远只有一个向量。Self-Attention 将赋予词向量"根据上下文动态变化"的能力。


## 段落 7：答疑 + 下一章预告

📍 参考 Cell 26-27

⏱ 时间分配：5 分钟

---

### 🎯 本段目标

- 回答学生遗留问题
- 预告下一章内容，建立知识连接

---

### 🗣 讲课话术

> 到这里本章的正式内容就结束了。在进入答疑之前，让我把下一章的内容预告一下。
>
> 今天学的 Embedding 有一个天生的缺陷：**同一个词永远只有一个向量**。"苹果好吃"的"苹果"和"苹果公司"的"苹果"，在 Embedding 矩阵里对应的是同一行。这叫做**一词多义问题**。
>
> Ch3 Self-Attention 将引入 Q、K、V 三个矩阵，让每个词的向量可以根据上下文动态调整。这是 Transformer 的核心机制，也是 GPT、BERT 等模型的灵魂。
>
> 好，大家有什么问题吗？

---

### ❓ 预判问题

**Q: 这章的代码我回去怎么复习？**

A: 重点关注三块代码：(1) Cell 6-7 理解查表等价性；(2) Cell 12-13 理解 Bigram 模型结构和训练循环；(3) Cell 19 自己重新实现一遍余弦相似度。Cell 27 提供了练习空间。

**Q: 推荐阅读什么延伸材料？**

A: Cell 26 列了两个：Mikolov 2013 年的 Word2Vec 原论文（看不懂公式没关系，看思路），以及 Jay Alammar 的图解 Word2Vec（强烈推荐，图文并茂）。


## 附录 A：时间快速参考表

| 时间 | 段落 | 核心动作 | 关键 Cell |
|:---|:---|:---|:---|
| 00:00 | 开场导入 | 运行环境准备 | Cell 4 |
| 05:00 | S1 查表矩阵 | 运行 Cell 6-7，验证等价性 | Cell 6, 7 |
| 20:00 | S2 Bigram 训练 | 数据准备->模型定义->训练->Loss 曲线 | Cell 10-14 |
| 45:00 | 休息 | 3 句话回顾 | - |
| 50:00 | S3 可视化 | PCA 散点图 | Cell 16, 17 |
| 60:00 | S4 余弦相似度 | **练习** + 相似度矩阵 | Cell 19, 20 |
| 75:00 | S5 对比 + 总结 | One-Hot vs Embedding | Cell 22 |
| 85:00 | 答疑 | 下一章预告 | Cell 26 |


## 附录 B：关键数据快速参考

### 源 Notebook 关键输出数值

| 数据项 | 数值 | 出处 |
|:---|:---|:---|
| PyTorch 版本 | 2.10.0+cu128 | Cell 4 |
| 训练文本总词数 | 52 | Cell 10 |
| 词表大小 | 28 | Cell 10 |
| 训练样本数 | 51 | Cell 11 |
| 嵌入维度 | 16 | Cell 12 |
| 模型总参数量 | 924 | Cell 12 |
| Epoch 100 Loss | 0.3044 | Cell 13 |
| Epoch 500 Loss（最终） | 0.2881 | Cell 13 |
| 训练耗时 | 0.22 秒 | Cell 13 |
| PCA 解释方差比 | [0.157, 0.132]，合计约 29% | Cell 16 |
| cos_sim(v, v) | 1.0000 | Cell 19 |
| cos_sim(v, -v) | -1.0000 | Cell 19 |
| find_similar('机') 最高 | '工' (0.3525) | Cell 19 |
| find_similar('学') 最高 | '界' (0.2223) | Cell 19 |
| One-Hot 维度 | 28 | Cell 22 |
| Embedding 维度 | 16 | Cell 22 |
| 维度压缩比 | 1.8x | Cell 22 |

### 参数计算明细

| 层 | 参数量计算 | 结果 |
|:---|:---|:---|
| Embedding(28, 16) | 28 x 16 | 448 |
| Linear(16, 28) 权重 | 16 x 28 | 448 |
| Linear(16, 28) 偏置 | 28 | 28 |
| **合计** | | **924** |


## 附录 C：应急预案

### 常见故障及处理

| 故障场景 | 症状 | 解决方案 |
|:---|:---|:---|
| **中文字体缺失** | 图表中中文显示为方框 | 运行 `plt.rcParams["font.sans-serif"] = ["SimHei"]` 或下载 NotoSansCJK 字体 |
| **sklearn 未安装** | `ModuleNotFoundError: No module named 'sklearn'` | `pip install scikit-learn` |
| **matplotlib 版本过低** | 图表样式异常 | `pip install --upgrade matplotlib` |
| **CUDA 不可用** | 不影响（本章全部在 CPU 运行） | 无需处理 |
| **训练 Loss 不下降** | Loss 保持在高位 | 检查学习率（应为 0.01），确认数据正确（X 和 Y 长度应为 51） |
| **Kernel 死亡/重启** | 变量丢失 | 从 Cell 4 开始重新运行所有代码 Cell |

### 时间不够的裁剪方案

| 剩余时间 | 裁剪策略 |
|:---|:---|
| 少 10 分钟 | 跳过 S5 One-Hot 对比（Cell 21-23），口头总结 |
| 少 20 分钟 | 跳过 S3 可视化（Cell 15-17），口头描述 PCA 图效果 |
| 少 30 分钟 | 跳过 S3 + S5，将余弦相似度练习改为课后作业 |

### 时间多余的扩展方案

| 多余时间 | 扩展内容 |
|:---|:---|
| 多 5 分钟 | 让学生修改 `embed_dim` 为 4 和 64，对比训练效果 |
| 多 10 分钟 | 让学生增加训练文本（Cell 10 的 text 变量），观察相似度变化 |
| 多 15 分钟 | 讨论 Cell 24 中的 3 个面试题，让学生先回答再对答案 |
